In [63]:
# %%
import pandas as pd
import numpy as np
import glob
import os
from IPython.display import display

print("Step 1: Loading Raw Data...")

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED, exist_ok=True)

# Define exactly which columns to read from the massive PUMS files
h_cols = ['SERIALNO', 'PUMA', 'FS', 'FFSP', 'WIF','HHLDRRAC1P', 'HHL',
          'ACCESSINET', 'FHISPEEDP', 'FLAPTOPP','FSMARTPHONP','FTABLETP',    # Technology access 
          'FKITP','FSINKP', 'FBATHP','FRWATP', 'FREFRP','FPLMP',              # Amenities
          'VEH', 'LNGI', 'GRPIP', 'RMSP', 'TEN', 'HINCP', 'NP','ACR', 
          'VALP','FINCP','FPARC','HHLDRAGEP','HHT2','NPP', 'HUPAC','R18','R65']

p_cols = ['SERIALNO', 'POVPIP', 'AGEP', 'DIS', 'SCHL', 'ESR', 'RAC1P', 'HISP','CIT']

h_files = glob.glob(os.path.join(DATA_RAW, "psam_h*.csv"))
p_files = glob.glob(os.path.join(DATA_RAW, "psam_p*.csv"))

if not h_files or not p_files:
    raise FileNotFoundError("Missing PUMS files in data/raw. Please run the download script first.")

df_h_raw = pd.concat([pd.read_csv(f, usecols=h_cols) for f in h_files], ignore_index=True)
df_p_raw = pd.concat([pd.read_csv(f, usecols=p_cols) for f in p_files], ignore_index=True)

print(f"Loaded Households: {df_h_raw.shape}")
print(f"Loaded People: {df_p_raw.shape}")

Step 1: Loading Raw Data...
Loaded Households: (74024, 35)
Loaded People: (157694, 9)


In [ ]:

# Handle Missing Values before feature engineering
df_h_raw['VEH'] = df_h_raw['VEH'].fillna(0)  
df_h_raw['GRPIP'] = df_h_raw['GRPIP'].fillna(0) 
df_h_raw['ACCESSINET'] = df_h_raw['ACCESSINET'].fillna(3) 
df_h_raw['LNGI'] = df_h_raw['LNGI'].fillna(1) 
df_h_raw['TEN'] = df_h_raw['TEN'].fillna(0) 

print("Step 2.1: Engineering Household-Level Features...")

# 1. ============================================= Economic stability  ============================================= 
# ---------- Income-based features ---------- 
# Log income (reduces skew)
df_h_raw['FINCP'] = pd.to_numeric(df_h_raw['FINCP'], errors='coerce').fillna(0)
df_h_raw['LOG_INCOME'] = np.log1p(df_h_raw['FINCP'])
df_h_raw['LOG_INCOME'] = df_h_raw['LOG_INCOME'].fillna(0)

# Low-income flag (relative poverty)
threshold = df_h_raw['FINCP'].median()
df_h_raw['LOW_INCOME_FLAG'] = (df_h_raw['FINCP'] < 0.5 * threshold).astype(int)

# ---------- Housing burden & stability ---------- 
# Rent burden indicator
df_h_raw['RENT_BURDEN'] = (df_h_raw['GRPIP'] > 50).astype(int)
# Ownership indicator
df_h_raw['IS_OWNER'] = df_h_raw['TEN'].isin([1, 2]).astype(int)
#df_h_raw['is_renter'] = df_h_raw['TEN'].isin([3, 4]).astype(int)

# ---------- Labor / earning capacity ---------- 
# Workers per capita
df_h_raw['WIF'] = pd.to_numeric(df_h_raw['WIF'], errors='coerce').fillna(0)
df_h_raw['NP'] = pd.to_numeric(df_h_raw['NP'], errors='coerce').fillna(0)
df_h_raw['WORKERS_PC'] = df_h_raw['WIF'] / df_h_raw['NP']
# Dependency ratio
df_h_raw['DEPENDENCY_RATIO'] = (df_h_raw['NP'] - df_h_raw['WIF']) / df_h_raw['NP']
# Wealth-to-income ratio
df_h_raw['VALP'] = pd.to_numeric(df_h_raw['VALP'], errors='coerce').fillna(0)
df_h_raw['WEALTH_INCOME_RATIO'] = df_h_raw['VALP'] / (df_h_raw['FINCP'] + 1)

# ---------- Interaction features ---------- 
df_h_raw['RENT_INCOME_INTERACTION'] = df_h_raw['GRPIP'] * df_h_raw['LOG_INCOME']
df_h_raw['WORKERS_INCOME_INTERACTION'] = df_h_raw['WIF'] * df_h_raw['FINCP']
df_h_raw['HOUSEHOLD_PRESSURE'] = df_h_raw['NP'] / (df_h_raw['WIF'] + 1)

# 2. Technology access (using access to internet, high speed, laptop, smartphone, tablet)
cols = ['FLAPTOPP', 'FSMARTPHONP', 'FTABLETP']
df_h_raw[cols] = df_h_raw[cols].fillna(0) # Treat NA as 0
# Device count (0 to 3)
df_h_raw['DEVICE_COUNT'] = df_h_raw[cols].sum(axis=1).astype(int) 
# Smartphone only = has smartphone, but no laptop and no tablet
df_h_raw['SMARTPHONE_ONLY'] = (
    (df_h_raw['FSMARTPHONP'] == 1) &
    (df_h_raw['FLAPTOPP'] == 0) &
    (df_h_raw['FTABLETP'] == 0)
).astype(int)

# Interaction features
df_h_raw['HAS_INTERNET'] = df_h_raw['ACCESSINET'].isin([1, 2]).astype(int)
df_h_raw['INTERNET_DEVICE_INTERACTION'] = (
    df_h_raw['HAS_INTERNET'] * df_h_raw['DEVICE_COUNT']
)


# 2.  =============================================  Amenities  ============================================= 
# Composite housing quality index
df_h_raw['HOUSING_QUALITY_INDEX'] = (
    (df_h_raw['FKITP'] == 0).astype(int) + (df_h_raw['FRWATP'] == 0).astype(int) + (df_h_raw['FREFRP'] == 0).astype(int) +
    (df_h_raw['FSINKP'] == 0).astype(int) + (df_h_raw['FPLMP'] == 0).astype(int) + (df_h_raw['FBATHP'] == 0).astype(int) +
    (df_h_raw['VEH'] == 1).astype(int)
)

# Amenity count
amenity_binary_cols = ['FKITP', 'FRWATP', 'FREFRP','FSINKP', 'FPLMP', 'FBATHP']
df_h_raw['AMENITY_COUNT'] = df_h_raw[amenity_binary_cols].sum(axis=1)

# Amenity deprivation indicators
df_h_raw['LOW_AMENITIES'] = (df_h_raw['AMENITY_COUNT'] <= 3).astype(int)

# 3.  ============================================= Language barriers  ============================================= 

# Convert LNGI
df_h_raw['LNGI'] = pd.to_numeric(df_h_raw['LNGI'], errors='coerce')
# Language barrier (non-English + isolated)
df_h_raw['LIMITED_ENGLISH'] = (df_h_raw['LNGI'] == 2).astype(int)

# Household language
# Example: create English vs non-English indicator
df_h_raw['ENGLISH_ONLY'] = (df_h_raw['HHL'] == 1).astype(int)


# 4.  =============================================  Household complexity  ============================================= 
# crowding features
df_h_raw['PERSONS_PER_ROOM'] = df_h_raw['NP'] / df_h_raw['RMSP'].replace(0, np.nan)
df_h_raw['CROWDED_HH'] = (df_h_raw['PERSONS_PER_ROOM'] > 2).astype(int)
# Multigenerational pressure
df_h_raw['CHILD_AND_SENIOR_HH'] = (
    (df_h_raw['R18'] == 1) &
    (df_h_raw['R65'] == 1)
).astype(int)
    

Step 2.1: Engineering Household-Level Features...


/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [65]:
# %%
print("Step 2.2: Engineering Person-Level Proxy Features...")

df_p = df_p_raw.copy()

# 1. Create binary flags for individuals
df_p['is_elderly'] = (df_p['AGEP'] >= 60).astype(int)
df_p['is_child'] = (df_p['AGEP'] < 18).astype(int)
df_p['is_disabled'] = (df_p['DIS'] == 1).astype(int) 
df_p['is_working'] = df_p['ESR'].isin([1, 2, 4, 5]).astype(int) 
df_p['is_minority'] = ((df_p['RAC1P'] != 1) | (df_p['HISP'] != 1)).astype(int)
df_p['is_citizen'] = (df_p['CIT'] == 1).astype(int) 
df_p['SCHL'] = df_p['SCHL'].fillna(0) 

# 2. Aggregate to Household Level (SERIALNO)
hh_agg = df_p.groupby('SERIALNO').agg(
    POVPIP=('POVPIP', 'first'),              # Crucial for defining the target later
    HAS_ELDERLY=('is_elderly', 'max'),       # present in household
    HAS_DISABLED=('is_disabled', 'max'),     
    NUM_CHILDREN=('is_child', 'sum'),        # present in household
    NUM_WORKING_ADULTS=('is_working', 'sum'), # present in household
    MAX_EDUCATION=('SCHL', 'max'),           
    IS_MINORITY_HH=('is_minority', 'max'),
    IS_CITIZEN_HH=('is_citizen', 'max'),    
).reset_index()

print("Aggregated person demographics to household level.")

Step 2.2: Engineering Person-Level Proxy Features...
Aggregated person demographics to household level.


In [66]:
# %%
print("Step 3: Merging, Filtering, and Creating the Blind Target...")

# 1. Merge
df_merged = pd.merge(df_h_raw, hh_agg, on='SERIALNO', how='inner')

# 2. Filter out Group Quarters and Imputed SNAP data
mask_standard = ~df_merged['SERIALNO'].astype(str).str.contains('GQ')
mask_real_snap = df_merged['FFSP'] != 1
df_clean = df_merged[mask_standard & mask_real_snap].copy()

# 3. Create Target Variable
# Class 1 (The Gap): Low income (<=130%) AND no SNAP (FS == 2)
# Class 0 (Not in Gap): Everyone else
df_clean['TARGET_GAP'] = ((df_clean['POVPIP'] <= 130) & (df_clean['FS'] == 2.0)).astype(int)
# poverty gap
df_clean['POVPIP'] = pd.to_numeric(df_clean['POVPIP'], errors='coerce')
df_clean['POVERTY_GAP'] = 130 - df_clean['POVPIP']
df_clean["ELIGIBLE"] = df_clean["POVPIP"].apply(lambda x: 1 if x <= 130 else 0)


# 5. FEATURE SELECTION  - Adding engineered features from both household and person level
# We purposefully DROP POVPIP, HINCP, FS, and FFSP here.
final_features = [
    'SERIALNO', 'PUMA', 'TARGET_GAP', 'HAS_ELDERLY', 'HAS_DISABLED', 
    'NUM_CHILDREN', 'NUM_WORKING_ADULTS', 'MAX_EDUCATION', 'IS_MINORITY_HH',
    'VEH', 'ACCESSINET', 'GRPIP', 'RMSP', 'NP',
    # 'LNGI',
    # new features
    'LOG_INCOME', 'LOW_INCOME_FLAG', 'RENT_BURDEN', 'POVERTY_GAP','IS_OWNER', 'WORKERS_PC',
    'WEALTH_INCOME_RATIO','RENT_INCOME_INTERACTION', 'WORKERS_INCOME_INTERACTION','HOUSEHOLD_PRESSURE',
    'DEVICE_COUNT', 'SMARTPHONE_ONLY','INTERNET_DEVICE_INTERACTION',
    'HOUSING_QUALITY_INDEX', 'AMENITY_COUNT', 'LOW_AMENITIES',
    'LIMITED_ENGLISH', 'ENGLISH_ONLY',
    'PERSONS_PER_ROOM', 'CROWDED_HH', 'CHILD_AND_SENIOR_HH'
]

df_final = df_clean[final_features].copy()

print(f"Final dataset ready: {df_final.shape[0]} total standard households.")
print(f"Target Distribution:\n{df_final['TARGET_GAP'].value_counts(normalize=True)*100}")

Step 3: Merging, Filtering, and Creating the Blind Target...
Final dataset ready: 61071 total standard households.
Target Distribution:
TARGET_GAP
0    91.929066
1     8.070934
Name: proportion, dtype: float64


In [67]:
# %%
print("Step 4: Generating Data Dictionary...")

doc_data = [
    {"Column": "SERIALNO", "Description": "Unique Household Identifier", "Type": "String"},
    {"Column": "PUMA", "Description": "Public Use Microdata Area (Geography Code)", "Type": "Categorical/Numeric"},
    {"Column": "TARGET_GAP", "Description": "TARGET: 1 if in the SNAP Gap, 0 otherwise", "Type": "Binary (0/1)"},

    ## ----------- Person features ----------- 
    {"Column": "HAS_ELDERLY", "Description": "1 if household has someone 60+", "Type": "Binary (0/1)"}, # already present in hh data
    {"Column": "NUM_CHILDREN", "Description": "Number of people under 18 in the household", "Type": "Numeric"}, # already present in hh data
    {"Column": "NUM_WORKING_ADULTS", "Description": "Number of employed adults in the household", "Type": "Numeric"}, # already present in hh data
    {"Column": "HAS_DISABLED", "Description": "1 if household has someone with a disability", "Type": "Binary (0/1)"}, 
    {"Column": "MAX_EDUCATION", "Description": "Highest educational attainment code in the household (1-24)", "Type": "Numeric (Ordinal)"},
    {"Column": "IS_MINORITY_HH", "Description": "1 if anyone in the household identifies as a racial/ethnic minority", "Type": "Binary (0/1)"},
    {"Column": "IS_CITIZEN_HH", "Description": "1 if anyone in the household is US citizen", "Type": "Character"},  
    
    ## ----------- Household variables and engineered features ----------- 
    
    # Economic stability
    # original variables
    {"Column": "VALP", "Description": "Property value", "Type": "Numeric"},
    {"Column": "FINCP", "Description": "Family income", "Type": "Numeric"},
    {"Column": "GRPIP", "Description": "Gross rent as a percentage of household income", "Type": "Numeric"}, # Housing burden
    {"Column": "TEN", "Description": "Housing tenure (1/2=Own, 3/4=Rent)", "Type": "Categorical"},
    {"Column": "NP", "Description": "Number of person in the household", "Type": "Numeric"},
    {"Column": "WIF", "Description": "Workers in family during the past 12 months", "Type": "Character"},
    {"Column": "ACR", "Description": "Lot size", "Type": "Character"},
    # engineered features
    {"Column": "LOG_INCOME", "Description": "Log-transformed family income (log(1 + FINCP))", "Type": "Numeric"},
    {"Column": "LOW_INCOME_FLAG", "Description": "Indicator for low income (1 if family income is below 50% of median, 0 otherwise)", "Type": "Numeric"},
    {"Column": "RENT_BURDEN", "Description": "Indicator for housing cost burden (1 if gross rent exceeds 30% of household income, 0 otherwise)", "Type": "Numeric"},
    {"Column": "IS_OWNER", "Description": "Indicator for home ownership (1 if housing tenure is owned, 0 if rented)", "Type": "Numeric"},
    {"Column": "WORKERS_PC", "Description": "Number of workers per household member (WIF divided by NP)", "Type": "Numeric"},
    {"Column": "DEPENDENCY_RATIO", "Description": "Ratio of non-working household members to total household size ((NP - WIF) / NP)", "Type": "Numeric"},
    {"Column": "WEALTH_INCOME_RATIO", "Description": "Ratio of property value to family income (VALP divided by FINCP)", "Type": "Numeric"},
    {"Column": "RENT_INCOME_INTERACTION", "Description": "Interaction between housing burden and income (GRPIP multiplied by log(1 + FINCP))", "Type": "Numeric"},
    {"Column": "WORKERS_INCOME_INTERACTION", "Description": "Interaction between number of workers and family income (WIF multiplied by FINCP)", "Type": "Numeric"},
    {"Column": "HOUSEHOLD_PRESSURE", "Description": "Household size relative to workers (NP divided by (WIF + 1))", "Type": "Numeric"},

    # Technology access
    # original variables
    {"Column": "ACCESSINET", "Description": "Internet access: 1/2=Yes, 3=No", "Type": "Categorical"},
    {"Column": "FHISPEEDP", "Description": "Broadband (high speed) Internet service allocation flag", "Type": "Character"},
    {"Column": "FLAPTOPP", "Description": "Laptop or desktop allocation flag", "Type": "Character"},
    {"Column": "FSMARTPHONP", "Description": "Smartphone allocation flag", "Type": "Character"},
    {"Column": "TABLET", "Description": "Tablet or other portable wireless computer allocation flag", "Type": "Character"},
    # engineered features   
    {"Column": "DEVICE_COUNT", "Description": "Count of devices", "Type": "Numeric"},
    {"Column": "SMARTPHONE_ONLY", "Description": "Has only smartphone", "Type": "Numeric"},
    {"Column": "INTERNET_DEVICE_INTERACTION", "Description": "Interaction between internet access and number of devices", "Type": "Numeric"},

    # Amenities
    # original variables
    {"Column": "VEH", "Description": "Number of vehicles available (0-6)", "Type": "Numeric"},
    {"Column": "FKITP", "Description": "Complete kitchen facilities allocation flag", "Type": "Character"},
    {"Column": "FRWATP", "Description": "Hot and cold running water allocation flag", "Type": "Character"},
    {"Column": "FREFRP", "Description": "Refrigerator allocation flag", "Type": "Character"},
    {"Column": "FSINKP", "Description": "Sink with a faucet allocation flag", "Type": "Character"},
    {"Column": "BATH", "Description": "Bathtub or shower", "Type": "Character"},
    {"Column": "FPLMP", "Description": "Complete plumbing facilities allocation flag", "Type": "Character"},
    # engineered features 
    {"Column": "HOUSING_QUALITY_INDEX", "Description": "Composite index of housing deprivation based on lack of basic amenities (kitchen, water, refrigerator, sink, plumbing, bath) and absence of vehicles; higher values indicate poorer housing conditions", "Type": "Numeric"},
    {"Column": "AMENITY_COUNT", "Description": "Total number of available basic household amenities including kitchen, running water, refrigerator, sink, plumbing, and bath facilities", "Type": "Numeric"},
    {"Column": "LOW_AMENITIES", "Description": "Indicator for limited household amenities (1 if AMENITY_COUNT is less than or equal to 3, 0 otherwise)", "Type": "Numeric"}, 
    
    # Language barriers
    {"Column": "HHL", "Description": "Household language", "Type": "Character"},
    {"Column": "LNGI", "Description": "Limited English speaking household: 1=Not limited, 2=Limited", "Type": "Categorical"},
    # engineered features 
    {"Column": "LIMITED_ENGLISH", "Description": "Indicator for one in the household 14 and over speaks English only or .speaks English 'very well'", "Type": "Numeric"},
    {"Column": "ENGLISH_ONLY", "Description": "Indicator for households with language English only", "Type": "Numeric"},
    
    # Household complexity and demographics
    {"Column": "RMSP", "Description": "Number of rooms in the housing unit", "Type": "Numeric"},
    {"Column": "NPP", "Description": "Grandparent headed household with no parent present", "Type": "Character"},
    {"Column": "FPARC", "Description": "Family presence and age of related children", "Type": "Character"},
    {"Column": "HHLDRAGEP", "Description": "Age of the householder", "Type": "Numeric"},
    {"Column": "HHT2", "Description": "Household/family type", "Type": "Character"},
    {"Column": "HHLDRRAC1P", "Description": "Recoded detailed race code of the householder", "Type": "Character"},
    {"Column": "HUPAC", "Description": "HH presence and age of children", "Type": "Character"},
    {"Column": "R18", "Description": "Presence of persons under 18 years in household", "Type": "Character"},
    {"Column": "R65", "Description": "Presence of persons 65 years and over in household", "Type": "Character"},
    # engineered features 
    {"Column": "PERSONS_PER_ROOM", "Description": "Average number of household members per room (NP divided by RMSP), used as a measure of housing crowding", "Type": "Numeric"},
    {"Column": "CROWDED_HH", "Description": "Indicator for crowded housing conditions (1 if PERSONS_PER_ROOM exceeds 2, 0 otherwise)", "Type": "Numeric"},
    {"Column": "CHILD_AND_SENIOR_HH", "Description": "Indicator for multigenerational household (1 if both children under 18 and seniors aged 65 or over are present, 0 otherwise)", "Type": "Numeric"},
    {"Column": "POVERTY_GAP", "Description": "Difference between 130 percent of the poverty threshold and household income as a percentage of poverty (130 minus POVPIP), indicating distance from the threshold", "Type": "Numeric"}

]

df_documentation = pd.DataFrame(doc_data)
display(df_documentation)

Step 4: Generating Data Dictionary...


,Column,Description,Type
0,SERIALNO,Unique Household Identifier,String
1,PUMA,Public Use Microdata Area (Geography Code),Categorical/Numeric
2,TARGET_GAP,"TARGET: 1 if in the SNAP Gap, 0 otherwise",Binary (0/1)
3,HAS_ELDERLY,1 if household has someone 60+,Binary (0/1)
4,NUM_CHILDREN,Number of people under 18 in the household,Numeric
...,...,...,...
57,R65,Presence of persons 65 years and over in house...,Character
58,PERSONS_PER_ROOM,Average number of household members per room (...,Numeric
59,CROWDED_HH,Indicator for crowded housing conditions (1 if...,Numeric
60,CHILD_AND_SENIOR_HH,Indicator for multigenerational household (1 i...,Numeric


In [68]:
# %%
print("Step 5: Sampling, Displaying, and Saving Results...")

df_gap = df_final[df_final['TARGET_GAP'] == 1]      
df_non_gap = df_final[df_final['TARGET_GAP'] == 0] 

sample_gap = df_gap.sample(n=min(20, len(df_gap)), random_state=42)
sample_non_gap = df_non_gap.sample(n=min(20, len(df_non_gap)), random_state=42)

print("\n THE GAP HOUSEHOLDS (TARGET_GAP = 1) - Sample of 20:")
display(sample_gap)

print("\n NON-GAP HOUSEHOLDS (TARGET_GAP = 0) - Sample of 20:")
display(sample_non_gap)

# Save processed data and dictionary
out_path = os.path.join(DATA_PROCESSED, "processed_all_households_blind.csv")
doc_path = os.path.join(DATA_PROCESSED, "data_dictionary_blind.csv")

df_final.to_csv(out_path, index=False)
df_documentation.to_csv(doc_path, index=False)

print(f"\n SUCCESS: Processed dataset saved to {out_path} ({df_final.shape[0]} rows)")
print(f" SUCCESS: Data Dictionary saved to {doc_path}")

Step 5: Sampling, Displaying, and Saving Results...

 THE GAP HOUSEHOLDS (TARGET_GAP = 1) - Sample of 20:


,SERIALNO,PUMA,TARGET_GAP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,...,SMARTPHONE_ONLY,INTERNET_DEVICE_INTERACTION,HOUSING_QUALITY_INDEX,AMENITY_COUNT,LOW_AMENITIES,LIMITED_ENGLISH,ENGLISH_ONLY,PERSONS_PER_ROOM,CROWDED_HH,CHILD_AND_SENIOR_HH
68858,2023HU1389337,7300,1,1,1,0,1,11.0,1,2.0,...,0,0,6,0.0,1,0,1,0.250000,0,0
52089,2023HU0699551,71002,1,1,0,0,0,21.0,0,0.0,...,0,0,6,0.0,1,0,1,0.333333,0,0
15488,2023HU0551209,601,1,0,0,0,0,20.0,0,2.0,...,0,0,6,0.0,1,0,1,0.200000,0,0
61885,2023HU1100499,70000,1,0,1,0,0,19.0,0,1.0,...,0,0,7,0.0,1,0,1,0.250000,0,0
53989,2023HU0775965,8702,1,1,0,0,0,24.0,0,2.0,...,0,0,6,0.0,1,0,1,0.222222,0,0
66934,2023HU1308924,17100,1,0,0,0,1,16.0,0,2.0,...,0,0,6,0.0,1,0,1,0.250000,0,0
46314,2023HU0453626,16500,1,0,1,0,0,20.0,0,0.0,...,0,0,6,0.0,1,0,1,1.000000,0,0
20880,2023HU0869987,1107,1,1,0,0,0,13.0,0,1.0,...,0,0,7,0.0,1,0,1,0.142857,0,0
64496,2023HU1206102,71001,1,0,0,1,1,19.0,0,1.0,...,0,0,7,0.0,1,0,1,0.222222,0,0
1846,2023HU0568474,103,1,1,1,0,2,21.0,1,1.0,...,0,0,7,0.0,1,0,0,0.800000,0,0



 NON-GAP HOUSEHOLDS (TARGET_GAP = 0) - Sample of 20:


,SERIALNO,PUMA,TARGET_GAP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,...,SMARTPHONE_ONLY,INTERNET_DEVICE_INTERACTION,HOUSING_QUALITY_INDEX,AMENITY_COUNT,LOW_AMENITIES,LIMITED_ENGLISH,ENGLISH_ONLY,PERSONS_PER_ROOM,CROWDED_HH,CHILD_AND_SENIOR_HH
52198,2023HU0704286,4103,0,1,1,0,1,21.0,0,3.0,...,0,0,6,0.0,1,0,1,0.222222,0,0
23311,2023HU1016450,1102,0,0,0,1,3,21.0,1,4.0,...,0,0,6,0.0,1,0,0,0.750000,0,0
68349,2023HU1368364,10701,0,0,0,0,3,21.0,0,4.0,...,0,0,6,0.0,1,0,1,0.444444,0,0
65385,2023HU1245941,1301,0,0,0,3,2,21.0,0,2.0,...,0,0,6,0.0,1,0,1,0.263158,0,0
3226,2023HU1198043,105,0,0,0,0,1,22.0,0,1.0,...,0,0,7,0.0,1,0,1,0.250000,0,0
63999,2023HU1186938,8300,0,0,0,0,1,16.0,0,2.0,...,0,0,6,0.0,1,0,1,0.142857,0,0
65072,2023HU1231210,6900,0,0,0,2,2,21.0,0,2.0,...,0,0,6,0.0,1,0,1,0.666667,0,0
49445,2023HU0587943,5906,0,1,0,0,1,22.0,0,2.0,...,0,0,6,0.0,1,0,0,0.222222,0,0
3161,2023HU1166248,102,0,0,0,0,3,22.0,0,2.0,...,0,0,6,0.0,1,0,1,0.750000,0,0
44294,2023HU0368151,60001,0,0,0,0,2,22.0,0,4.0,...,0,0,6,0.0,1,0,1,0.222222,0,0



 SUCCESS: Processed dataset saved to ../data/processed/processed_all_households_blind.csv (61071 rows)
 SUCCESS: Data Dictionary saved to ../data/processed/data_dictionary_blind.csv
